# 2. 인공지능기본법 Embedding과 Vector Retrieval

- 참고 실습: `track1_core/03-2_embedding_vector_retrieval.ipynb`
- 입력: HWPX 법률 문서
- 검색 단위: 조문 또는 항·호·목 기반 Child Chunk


## 1. 학습목표

이 실습을 완료하면 다음을 수행할 수 있다.

1. Embedding이 텍스트를 벡터 공간에 표현하는 방식을 설명할 수 있다.
2. 코사인 유사도 점수의 의미와 한계를 설명하고 검색 순위에 활용할 수 있다.
3. Vector Store의 Top-k 검색과 Metadata Filter를 사용해 검색 파이프라인을 구성할 수 있다.
4. 검색어의 표현 방식(짧음/구체적, 동의어, 동일 키워드의 다른 의미)이 검색 결과에
   미치는 영향을 관찰할 수 있다.


## 2. 문제 상황

법률 질의는 표현이 달라도 같은 법적 개념을 가리킬 수 있다. 예를 들어 “생성형 AI
표시”와 “인공지능 생성물 고지”는 제31조와 관련된다. Embedding 검색으로 의미상
가까운 조문을 찾되, 본칙/부칙·장·조문 metadata Filter로 후보 범위를 통제한다.


## 3. 핵심 개념

### 3.1 정의

Embedding은 텍스트를 고정된 차원의 실수 벡터로 변환하는 것이다. 이렇게 만들어진
벡터 공간에서는 의미가 비슷한 텍스트일수록 벡터 사이의 거리가 가깝다.

### 3.2 개념이 필요한 이유

키워드 검색은 문서에 질문과 똑같은 단어가 있어야만 그 문서를 찾을 수 있다.
그러나 사용자는 같은 의미를 다른 단어로 표현하는 경우가 많다("생성형 AI" vs
"생성형 인공지능"). Embedding 기반 검색은 단어가 달라도 의미가 비슷하면 찾아낼 수 있어,
검색어 표현에 덜 민감한 검색이 가능해진다.

### 3.3 주요 구성요소

| 구성요소 | 의미 |
|---|---|
| Vector Store | 문서 Embedding과 metadata를 저장하고 유사도 검색을 수행하는 구성요소 |
| 문서/질문 Embedding | 문서와 질문을 같은 벡터 공간으로 변환한 결과 |
| 코사인 유사도 | 질문과 문서 벡터가 얼마나 비슷한지를 비교하는 순위 점수 |
| 순위(rank) | 코사인 유사도가 높은 순서대로 매긴 순번 |
| Top-k | 순위가 높은 상위 k개 결과만 선택하는 것 |
| Metadata Filter | `document_id`, `section` 등 조건으로 검색 후보군을 미리 좁히는 것 |

### 3.4 동작 과정

```text
샘플 문서 → Document 변환 → Vector Store에 저장
                                  ↓
질문 → similarity_search_with_relevance_scores(query, k, filter)
                                  ↓
      Vector Store가 Embedding·유사도·정렬·Top-k 처리
                                  ↓
                    검색 결과 DataFrame
```

### 3.5 코드와 개념의 대응 관계

| 코드 요소 | 구현 개념 |
|---|---|
| `get_chroma_store()` | 공통 경로에서 디스크 기반 Chroma Collection 생성·재사용 |
| `add_documents_if_empty()` | 기존 Collection이 비었을 때만 문서와 metadata 저장 |
| `similarity_search_with_relevance_scores(...)` | 질문 Embedding·유사도·정렬·Top-k 검색 |
| `make_metadata_filter()` | Vector Store에 전달할 Metadata Filter |
| `search()` | 검색 조건을 Vector Store에 전달하고 결과 형식을 변환 |
| `build_result_dataframe()` | 검색 결과 DataFrame 출력 |

### 3.6 유사 개념과의 차이

**반드시 교정해야 할 오해**

```text
유사도 점수 0.85
≠ 정답일 확률 85%
```

코사인 유사도는 질문 벡터와 문서 벡터가 얼마나 비슷한지를 비교하는 점수일 뿐,
그 문서가 질문에 대한 올바른 답을 담고 있다는 보장이 아니다. 유사도 점수는 순위를
매기는 데 쓰는 상대적 지표이지, 정답일 확률 같은 절대적 지표가 아니다.

**키워드 검색 vs Embedding 검색**

| 구분 | 키워드 검색 | Embedding 검색 |
|---|---|---|
| 판단 기준 | 문자열이 정확히 일치하는가 | 벡터 방향이 얼마나 비슷한가 |
| 동의어 인식 | 불가능 | 가능(모델 성능에 따라 다름) |
| 계산 비용 | 낮음 | Embedding 계산 비용 발생 |
| 결과 해석 | "포함되어 있다/없다"로 명확함 | 점수는 상대적 순위 신호일 뿐 |

### 3.7 사용 시점과 적용 조건

문서량이 많고 사용자가 다양한 표현으로 질문할 가능성이 높다면 Embedding 검색이
유리하다. 반대로 정확한 코드, ID, 고유명사처럼 정확히 일치해야 의미가 있는 검색은
키워드 검색이 더 적합할 수 있다.

### 3.8 한계와 주의사항

- 코사인 유사도 값 자체는 모델과 문서 집합에 따라 상대적이며, 절대적인 기준값으로
  삼을 수 없다.
- Metadata Filter를 너무 좁게 걸면 실제 정답이 담긴 문서가 후보군에서 아예
  제외될 수 있다.
- Top-k가 너무 작으면 관련 문서를 놓치고, 너무 크면 무관한 문서까지 함께
  전달되어 비용이 늘어난다.

### 3.9 자주 발생하는 오해

"유사도 점수가 높으면 그 문서가 정답"이라는 오해가 가장 흔하다. 실제로는 유사도
점수가 높아도 질문과 무관한 이유로 벡터가 가까워졌을 수 있고, 반대로 진짜 정답을
담은 문서의 점수가 더 낮게 나올 수도 있다. 유사도 점수는 후보를 좁히는 신호일
뿐이며, 실제로 정답인지는 문서 본문을 확인해야 한다.

### 3.10 핵심 정리

- Embedding은 텍스트를 벡터로 표현해 의미 기반 검색을 가능하게 한다.
- Vector Store가 Embedding 저장, 유사도 계산, 정렬, Top-k를 처리하므로 애플리케이션에서
  같은 기능을 다시 구현할 필요가 없다.
- 유사도 점수는 검색 순위를 위한 상대적 지표이며, 정답일 확률이 아니다.
- Metadata Filter는 유사도 검색 전에 후보군 자체를 좁히는 절차다.
- 검색 결과의 실제 정확성은 유사도 점수가 아니라 문서 본문을 확인해야 판단할 수
  있다.


## 4. 실행 구조

```text
HWPX → 법률 계층 파싱 → 조문/항·호·목 Chunk
     → Chroma(aitrust_law_chunks_v1)
     → 의미 검색 + 본칙/부칙·장·조문 Filter
     → 순위·유사도·법률 인용 metadata 확인
```


## 5. 환경 설정

반복되는 경로 탐색과 모델 생성 코드는 `src/agentic_ai` 공통 모듈에서 관리한다.
아래 셀에서는 이 Notebook에 필요한 표준 라이브러리와 공통 기능만 불러온다.

> 처음 실행하거나 환경 오류가 발생하면 프로젝트 루트의
> `00_environment_check.ipynb`를 먼저 실행한다.


In [ ]:
import re
import zipfile
from xml.etree import ElementTree as ET

from langchain_core.documents import Document

from agentic_ai.config import get_settings
from agentic_ai.logging_utils import save_log
from agentic_ai.models import get_embedding_model
from agentic_ai.notebook_utils import print_environment_summary
from agentic_ai.paths import CHROMA_DIR, OUTPUT_DIR, PROJECT_ROOT, data_path
from agentic_ai.retrieval_utils import add_documents_if_empty, get_chroma_store
from agentic_ai.tools import search_keyword

settings = get_settings()
print_environment_summary(settings, needs_embedding_model=True)


## 6. 유사도 점수 먼저 읽기

이 실습에서는 코사인 유사도 공식을 직접 구현하지 않는다. 검색 결과에서 점수가 클수록
질문과 문서가 상대적으로 더 유사해 높은 순위를 받는다는 점에 집중한다. 아래 예시처럼
같은 후보군 안에서 점수를 비교해 순서를 정하며, 점수 자체는 정답 확률을 의미하지 않는다.


In [ ]:
score_examples = [
    {"후보": "문서 A", "유사도": 0.71},
    {"후보": "문서 B", "유사도": 0.92},
    {"후보": "문서 C", "유사도": 0.34},
]

ranked_examples = sorted(score_examples, key=lambda item: item["유사도"], reverse=True)
for rank, example in enumerate(ranked_examples, start=1):
    print(f"{rank}위 {example['후보']}: {example['유사도']:.2f}")

print("주의: 가장 높은 점수도 '정답일 확률'은 아닙니다.")


## 7. 단계별 구현

### 7.1 HWPX 법률 Chunk 생성

노트북 1과 동일한 파서와 계층 Chunker를 포함해 이 노트북만 실행해도 독립적으로
HWPX를 로드할 수 있다.


In [ ]:
DATA_PATH = data_path(
    "samples",
    "인공지능 발전과 신뢰 기반 조성 등에 관한 기본법(법률)(제20676호)(20260122).hwpx",
    must_exist=True,
)
LAW_ID = "ai-trust-basic-act-20676"
LAW_NAME = "인공지능 발전과 신뢰 기반 조성 등에 관한 기본법"
SOURCE_FILE = DATA_PATH.name
COLLECTION_NAME = "aitrust_law_chunks_v1"

HP_NS = "http://www.hancom.co.kr/hwpml/2011/paragraph"
XML_NS = {"hp": HP_NS}


def load_hwpx_paragraphs(file_path) -> list[str]:
    """HWPX의 section XML에서 화면에 표시되는 문단 텍스트를 순서대로 추출한다."""
    paragraphs: list[str] = []
    with zipfile.ZipFile(file_path) as archive:
        section_names = sorted(
            name
            for name in archive.namelist()
            if re.fullmatch(r"Contents/section\d+\.xml", name)
        )
        if not section_names:
            raise ValueError(f"HWPX 본문 section XML을 찾을 수 없습니다: {file_path}")

        for section_name in section_names:
            root = ET.fromstring(archive.read(section_name))
            for paragraph in root.findall(".//hp:p", XML_NS):
                raw_text = "".join(
                    node.text or "" for node in paragraph.findall(".//hp:t", XML_NS)
                )
                normalized = re.sub(r"\s+", " ", raw_text).strip()
                if normalized:
                    paragraphs.append(normalized)
    return paragraphs


RAW_PARAGRAPHS = load_hwpx_paragraphs(DATA_PATH)
RAW_TEXT = "\n".join(RAW_PARAGRAPHS)
effective_date_match = re.search(r"\[시행\s+([^\]]+)\]", RAW_TEXT)
LAW_EFFECTIVE_DATE = effective_date_match.group(1) if effective_date_match else "확인 필요"

first_chapter_index = next(
    index for index, paragraph in enumerate(RAW_PARAGRAPHS)
    if re.match(r"^제\d+장\s+", paragraph)
)
LAW_BODY_PARAGRAPHS = RAW_PARAGRAPHS[first_chapter_index:]
document_text = "\n".join(LAW_BODY_PARAGRAPHS)

print(f"입력 파일: {DATA_PATH}")
print(f"전체 HWPX 문단: {len(RAW_PARAGRAPHS)}개")
print(f"법령 본문 문단: {len(LAW_BODY_PARAGRAPHS)}개 / {len(document_text):,}자")
print(f"시행일: {LAW_EFFECTIVE_DATE}")

CHAPTER_RE = re.compile(r"^제(?P<number>\d+)장\s+(?P<title>.+)$")
ADDENDUM_RE = re.compile(r"^부칙(?:\s|<|$)")
ARTICLE_RE = re.compile(
    r"^제(?P<number>\d+)조(?:의(?P<sub_number>\d+))?"
    r"\((?P<title>[^)]+)\)\s*(?P<body>.*)$"
)
PARAGRAPH_RE = re.compile(r"^[①②③④⑤⑥⑦⑧⑨⑩⑪⑫⑬⑭⑮⑯⑰⑱⑲⑳]")
ITEM_RE = re.compile(r"^\d+(?:의\d+)?\.\s*")
SUBITEM_RE = re.compile(r"^[가-하]\.\s*")


def parse_law_articles(paragraphs: list[str]) -> list[dict]:
    """장·조문·부칙 경계를 인식해 법률을 조문 레코드로 변환한다."""
    articles: list[dict] = []
    chapter = ""
    scope = "본칙"
    current: dict | None = None

    def flush_current() -> None:
        nonlocal current
        if current is None:
            return
        current["text"] = "\n".join(current["lines"])
        current["length"] = len(current["text"])
        articles.append(current)
        current = None

    for paragraph in paragraphs:
        chapter_match = CHAPTER_RE.match(paragraph)
        if chapter_match:
            flush_current()
            scope = "본칙"
            chapter = paragraph
            continue

        if ADDENDUM_RE.match(paragraph):
            flush_current()
            scope = "부칙"
            chapter = paragraph
            continue

        article_match = ARTICLE_RE.match(paragraph)
        if article_match:
            flush_current()
            number = article_match.group("number")
            sub_number = article_match.group("sub_number") or ""
            article_label = f"제{number}조" + (f"의{sub_number}" if sub_number else "")
            scope_key = "main" if scope == "본칙" else "addendum"
            article_key = number + (f"-{sub_number}" if sub_number else "")
            article_id = f"{LAW_ID}-{scope_key}-article-{article_key}"
            title = article_match.group("title").strip()
            current = {
                "article_id": article_id,
                "document_id": LAW_ID,
                "law_name": LAW_NAME,
                "scope": scope,
                "chapter": chapter,
                "article": article_label,
                "article_number": article_key,
                "article_title": title,
                "section": f"{article_label}({title})",
                "source_file": SOURCE_FILE,
                "effective_date": LAW_EFFECTIVE_DATE,
                "lines": [paragraph],
            }
        elif current is not None:
            current["lines"].append(paragraph)

    flush_current()
    return articles


ARTICLES = parse_law_articles(LAW_BODY_PARAGRAPHS)
MAIN_ARTICLES = [article for article in ARTICLES if article["scope"] == "본칙"]
ADDENDUM_ARTICLES = [article for article in ARTICLES if article["scope"] == "부칙"]

print(f"조문 수: {len(ARTICLES)}개 (본칙 {len(MAIN_ARTICLES)}개 / 부칙 {len(ADDENDUM_ARTICLES)}개)")
print("첫 조문:", MAIN_ARTICLES[0]["section"])
print("마지막 본칙 조문:", MAIN_ARTICLES[-1]["section"])

def _body_lines(article: dict) -> list[str]:
    """첫 줄의 조문 표제는 제거하고 조문 본문만 반환한다."""
    first_match = ARTICLE_RE.match(article["lines"][0])
    first_body = first_match.group("body").strip() if first_match else article["lines"][0]
    return ([first_body] if first_body else []) + article["lines"][1:]


def _split_groups(lines: list[str], marker: re.Pattern) -> tuple[list[str], list[list[str]]]:
    """표지 문맥과 marker로 시작하는 연속 그룹을 분리한다."""
    prelude: list[str] = []
    groups: list[list[str]] = []
    current: list[str] | None = None
    for line in lines:
        if marker.match(line):
            if current:
                groups.append(current)
            current = [line]
        elif current is None:
            prelude.append(line)
        else:
            current.append(line)
    if current:
        groups.append(current)
    return prelude, groups


def _semantic_units(article: dict, budget: int) -> list[tuple[str, list[str]]]:
    """긴 조문을 항→호→목 경계로만 내려가며 의미 단위로 분리한다."""
    body = _body_lines(article)
    if not body:
        return [("article", [])]

    if any(PARAGRAPH_RE.match(line) for line in body):
        primary_type, primary_marker, secondary_type, secondary_marker = (
            "paragraph", PARAGRAPH_RE, "item", ITEM_RE
        )
    elif any(ITEM_RE.match(line) for line in body):
        primary_type, primary_marker, secondary_type, secondary_marker = (
            "item", ITEM_RE, "subitem", SUBITEM_RE
        )
    else:
        return [("article_part", body)]

    shared_intro, primary_groups = _split_groups(body, primary_marker)
    units: list[tuple[str, list[str]]] = []
    for primary_group in primary_groups:
        candidate = shared_intro + primary_group
        if len("\n".join(candidate)) <= budget:
            units.append((primary_type, candidate))
            continue

        local_intro, secondary_groups = _split_groups(primary_group, secondary_marker)
        if not secondary_groups:
            for line in primary_group:
                units.append((primary_type, shared_intro + [line]))
            continue

        for secondary_group in secondary_groups:
            secondary_candidate = shared_intro + local_intro + secondary_group
            if len("\n".join(secondary_candidate)) <= budget:
                units.append((secondary_type, secondary_candidate))
                continue

            tertiary_intro, tertiary_groups = _split_groups(secondary_group, SUBITEM_RE)
            if tertiary_groups:
                for tertiary_group in tertiary_groups:
                    units.append(("subitem", shared_intro + local_intro + tertiary_intro + tertiary_group))
            else:
                for line in secondary_group:
                    units.append((secondary_type, shared_intro + local_intro + [line]))
    return units or [("article_part", body)]


def _base_metadata(article: dict) -> dict:
    return {
        "document_id": article["document_id"],
        "law_name": article["law_name"],
        "scope": article["scope"],
        "chapter": article["chapter"],
        "article": article["article"],
        "article_number": article["article_number"],
        "article_title": article["article_title"],
        "section": article["section"],
        "source_file": article["source_file"],
        "effective_date": article["effective_date"],
        "hierarchy_path": f"{article['scope']} > {article['chapter']} > {article['section']}",
    }


def _context_prefix(article: dict) -> str:
    return f"{LAW_NAME}\n{article['scope']} | {article['chapter']}\n{article['section']}"


def build_parent_documents(articles: list[dict]) -> list[dict]:
    """답변 시 전체 조문 맥락으로 확장할 Parent 문서를 만든다."""
    parents = []
    for article in articles:
        text = f"{LAW_NAME}\n{article['scope']} | {article['chapter']}\n{article['text']}"
        parents.append({
            "doc_id": article["article_id"],
            **_base_metadata(article),
            "chunk_type": "article_parent",
            "parent_id": "",
            "text": text,
            "length": len(text),
        })
    return parents


def build_retrieval_documents(articles: list[dict], max_chars: int = 1000) -> list[dict]:
    """짧은 조문은 그대로, 긴 조문은 법률 계층 경계로 분할한다."""
    retrieval_documents: list[dict] = []
    for article in articles:
        prefix = _context_prefix(article)
        body_text = "\n".join(_body_lines(article))
        contextual_full_text = prefix + (f"\n{body_text}" if body_text else "")
        if len(contextual_full_text) <= max_chars:
            retrieval_documents.append({
                "doc_id": article["article_id"],
                **_base_metadata(article),
                "chunk_type": "article",
                "parent_id": "",
                "text": contextual_full_text,
                "length": len(contextual_full_text),
            })
            continue

        budget = max(200, max_chars - len(prefix) - 1)
        units = _semantic_units(article, budget)
        for index, (chunk_type, unit_lines) in enumerate(units, start=1):
            text = prefix + ("\n" + "\n".join(unit_lines) if unit_lines else "")
            retrieval_documents.append({
                "doc_id": f"{article['article_id']}-part-{index:02d}",
                **_base_metadata(article),
                "chunk_type": chunk_type,
                "parent_id": article["article_id"],
                "text": text,
                "length": len(text),
            })
    return retrieval_documents


PARENT_DOCUMENTS = build_parent_documents(ARTICLES)
RETRIEVAL_DOCUMENTS = build_retrieval_documents(ARTICLES, max_chars=1000)
SAMPLE_DOCUMENTS = RETRIEVAL_DOCUMENTS
PARENT_BY_ID = {document["doc_id"]: document for document in PARENT_DOCUMENTS}

long_article_children = [document for document in RETRIEVAL_DOCUMENTS if document["parent_id"]]
print(f"Parent 조문: {len(PARENT_DOCUMENTS)}개")
print(f"검색 Chunk: {len(RETRIEVAL_DOCUMENTS)}개 / 장문 조문 Child: {len(long_article_children)}개")
print("Chunk 유형:", sorted({document["chunk_type"] for document in RETRIEVAL_DOCUMENTS}))


### 7.2 Vector Store 구성

문서를 `Document`로 변환해 Vector Store에 추가한다. 문서 Embedding 생성과 저장은
Vector Store가 담당하며, 애플리케이션에서 Embedding 목록을 직접 관리하지 않는다.

> 이 실습은 로컬 디스크의 `outputs/vectorstore/chroma/`에 Chroma DB를 저장한다.
> 노트북을 다시 실행해도 같은 컬렉션과 문서를 재사용하며, 동일한 문서 ID는 새로
> 중복 추가하지 않고 갱신한다.


In [ ]:
embedding_model = get_embedding_model()
# 검색 대상 본문은 page_content에, 필터·출처 정보는 metadata에 분리해 저장한다.
VECTOR_DOCUMENTS = [
    Document(
        page_content=record["text"],
        metadata={key: value for key, value in record.items() if key != "text"},
    )
    for record in SAMPLE_DOCUMENTS
]

vector_store = get_chroma_store(
    COLLECTION_NAME,
    embedding_model=embedding_model,
)
# 영속 Collection이 비어 있을 때만 추가해 Notebook 재실행 시 중복 적재를 막는다.
documents_added, document_count = add_documents_if_empty(
    vector_store,
    VECTOR_DOCUMENTS,
    ids=[record["doc_id"] for record in SAMPLE_DOCUMENTS],
)

action = "초기화" if documents_added else "기존 Collection 재사용"
print(f"Chroma Collection: {COLLECTION_NAME}")
print(f"디스크 저장 경로: {CHROMA_DIR}")
print(f"처리 결과: {action} ({document_count}개 문서)")


### 7.3 유사도 검색

질문을 문자열로 전달하면 Vector Store가 질문 Embedding, 유사도 계산, 정렬, Top-k
선택을 처리한다. 반환값은 `(Document, score)` 튜플 목록이다.


In [ ]:
sample_matches = vector_store.similarity_search_with_relevance_scores(
    "고영향 인공지능 사업자가 이행해야 하는 안전성과 신뢰성 확보 조치",
    k=5,
)
for document, score in sample_matches:
    print(f"score={score:.4f} | {document.metadata['section']} | {document.metadata['chunk_type']}")


### 7.4 검색 결과 변환

Vector Store가 반환한 `Document`와 점수를 실습에서 관찰하기 쉬운 dict 형태로
변환하고, 반환 순서대로 순위를 부여한다.


In [ ]:
def to_result_records(matches: list[tuple[Document, float]]) -> list[dict]:
    """Vector Store 검색 결과를 순위가 포함된 dict 목록으로 변환한다."""
    # Vector Store가 유사도순으로 반환한 순서를 유지하며 1부터 rank를 붙인다.
    return [
        {
            **document.metadata,
            "text": document.page_content,
            "similarity": score,
            "rank": rank,
        }
        for rank, (document, score) in enumerate(matches, start=1)
    ]


### 7.5 Metadata Filter

Chroma의 `filter` 인자에 전달할 metadata 조건 dict를 만든다. 필터는 유사도 검색 전에
DB 내부 후보 문서를 좁히며, 점수 계산과 정렬은 Chroma가 담당한다.


In [ ]:
def make_metadata_filter(
    document_id: str | None = None,
    scope: str | None = None,
    chapter: str | None = None,
    article: str | None = None,
) -> dict | None:
    conditions = {
        "document_id": document_id,
        "scope": scope,
        "chapter": chapter,
        "article": article,
    }
    active = [{key: {"$eq": value}} for key, value in conditions.items() if value is not None]
    if not active:
        return None
    return active[0] if len(active) == 1 else {"$and": active}


print(make_metadata_filter(scope="본칙", article="제34조"))


### 7.6 Vector Store 검색 함수

`search()`는 조건과 `k`를 Vector Store에 전달하고 결과 형식만 변환한다.
Embedding 생성, 유사도 계산, 정렬, Top-k 선택은 직접 구현하지 않는다.


In [ ]:
def search(
    query: str,
    k: int = 5,
    document_id: str | None = None,
    scope: str | None = None,
    chapter: str | None = None,
    article: str | None = None,
) -> list[dict]:
    if k < 1:
        raise ValueError("k는 1 이상이어야 합니다.")
    metadata_filter = make_metadata_filter(
        document_id=document_id,
        scope=scope,
        chapter=chapter,
        article=article,
    )
    matches = vector_store.similarity_search_with_relevance_scores(
        query,
        k=k,
        filter=metadata_filter,
    )
    return to_result_records(matches)


### 7.7 Metadata Filter 실행

`document_id` 조건을 Vector Store 검색에 전달해 후보 문서가 실제로 좁혀지는지
확인한다.


In [ ]:
trust_chapter = next(chapter for chapter in {doc["chapter"] for doc in SAMPLE_DOCUMENTS} if chapter.startswith("제4장"))
filtered_demo = search(
    "생성형 인공지능 표시와 고영향 인공지능 의무",
    k=len(SAMPLE_DOCUMENTS),
    scope="본칙",
    chapter=trust_chapter,
)
print(f"제4장 Filter 결과: {len(filtered_demo)}개")
print({result["chapter"] for result in filtered_demo})


### 7.8 검색 결과 DataFrame 출력

Vector Store 검색 결과를 DataFrame으로 변환해 순위, metadata, 유사도 점수와 본문을
한눈에 비교한다.


In [ ]:
import pandas as pd


def build_result_dataframe(query: str, results: list[dict]) -> pd.DataFrame:
    rows = [{
        "query": query,
        "rank": result["rank"],
        "doc_id": result["doc_id"],
        "scope": result["scope"],
        "chapter": result["chapter"],
        "article": result["article"],
        "article_title": result["article_title"],
        "chunk_type": result["chunk_type"],
        "similarity": round(result["similarity"], 4),
        "text": result["text"],
    } for result in results]
    return pd.DataFrame(rows)


BASELINE_QUERY = "고영향 인공지능 사업자의 안전성 및 신뢰성 확보 의무"
baseline_results = search(BASELINE_QUERY, k=5)
build_result_dataframe(BASELINE_QUERY, baseline_results)


## 8. 실행 결과 관찰

`baseline_results`의 순위와 유사도 점수, 그리고 실제 본문을 함께 확인한다.


In [ ]:
for r in baseline_results:
    print(f"[{r['rank']}위] similarity={r['similarity']:.4f} | section={r['section']} | {r['text']}")


**결과 해석**: 순위(`rank`)는 유사도 점수(`similarity`)를 기준으로 매겨진다. 점수
자체의 절대값보다는, 어떤 문서가 다른 문서보다 상대적으로 질문과 가까운지를
비교하는 데 의미가 있다.


## 9. 비교 실험

### 9.1 Top-1과 Top-3


In [ ]:
query_a = "생성형 인공지능 결과물 표시 의무는 무엇인가요?"
top1 = search(query_a, k=1)
top5 = search(query_a, k=5)
print("Top-1:")
for result in top1:
    print(result["article"], result["article_title"], result["similarity"])
print("Top-5:")
for result in top5:
    print(result["rank"], result["article"], result["article_title"], result["similarity"])


**관찰**: Top-1만 보면 가장 유사도가 높은 문서 하나만 확인할 수 있지만, 실제 정답에
필요한 정보가 2~3위 문서에 나뉘어 있을 수도 있다. Top-k를 늘리면 놓칠 수 있는 정보를
줄일 수 있지만, 그만큼 무관한 문서가 섞일 가능성도 커진다.


### 9.2 짧은 검색어와 구체적인 검색어


In [ ]:
short_query = "표시 의무"
specific_query = "생성형 인공지능으로 만든 결과물에 인공지능 생성물임을 표시해야 하는 의무"
short_results = search(short_query, k=5)
specific_results = search(specific_query, k=5)

for label, results in [("짧은 검색어", short_results), ("구체적 검색어", specific_results)]:
    print(label)
    for result in results:
        print(result["rank"], result["article"], result["article_title"], round(result["similarity"], 4))


**관찰**: 짧은 검색어는 여러 문서와 폭넓게 비슷하게 나올 수 있어 순위 사이의 점수
차이가 작을 수 있다. 구체적인 검색어는 관련 문서와 무관한 문서 사이의 유사도 차이가
더 뚜렷하게 나타나는 경향이 있다.


### 9.3 동의어


In [ ]:
legal_term_query = "생성형 인공지능 결과물 표시"
common_term_query = "생성형 AI 콘텐츠 라벨링"
legal_term_results = search(legal_term_query, k=5)
common_term_results = search(common_term_query, k=5)

print("법률 용어 검색:", [(r["article"], r["article_title"]) for r in legal_term_results])
print("일상 용어 검색:", [(r["article"], r["article_title"]) for r in common_term_results])


**관찰**: 법률 원문의 “생성형 인공지능”과 일상 표현 “생성형 AI”, “라벨링”은
문자열이 다르다. Embedding 검색은 의미가 가까운 제31조를 후보로 올릴 수 있지만,
최종 판단은 반드시 조문 본문과 인용 metadata를 확인해야 한다.


### 9.4 동일 키워드의 다른 의미


In [ ]:
article31 = next(doc for doc in SAMPLE_DOCUMENTS if doc["scope"] == "본칙" and doc["article"] == "제31조")
article32 = next(doc for doc in SAMPLE_DOCUMENTS if doc["scope"] == "본칙" and doc["article"] == "제32조")

print("제31조 '의무' 키워드:", search_keyword(article31["text"], "의무"))
print("제32조 '의무' 키워드:", search_keyword(article32["text"], "의무"))

obligation_query = "인공지능 생성물임을 이용자에게 알리는 의무"
obligation_results = search(obligation_query, k=10)
for result in obligation_results:
    if result["article"] in {"제31조", "제32조"}:
        print(result["rank"], result["article"], result["article_title"])


**관찰**: `의무`라는 같은 단어가 제31조의 투명성 의무와 제32조의 안전성 의무에
모두 나타난다. 키워드 포함 여부만으로는 구별하기 어렵고, 질의 문맥과 조문 제목을
함께 봐야 한다.


### 9.5 Metadata Filter 적용 전후


In [ ]:
query_b = "시행일과 시행 준비행위"
without_filter = search(query_b, k=5)
addendum_only = search(query_b, k=5, scope="부칙")

print("Filter 미적용:")
for result in without_filter:
    print(result["rank"], result["scope"], result["section"])
print("부칙 Filter 적용:")
for result in addendum_only:
    print(result["rank"], result["scope"], result["section"])


**관찰**: “시행일”은 본칙의 개별 시행일 문구와 부칙 제1조 모두에 등장할 수 있다.
`scope="부칙"` Filter를 사용하면 부칙 조문만 후보로 제한해 법률 구조에 맞는 검색을
수행할 수 있다.


## 10. 실패 실험과 교정: 유사도 점수를 정답 확률로 오해하기

실제 문서 대신 통제된 검색 결과를 사용해, Vector Store가 반환한 유사도 점수가 높아도
그 문서가 질문의 정답을 담고 있다고 보장할 수 없음을 확인한다.


In [ ]:
controlled_results = [
    {"doc_id": "A", "similarity": 0.95, "contains_answer": False},
    {"doc_id": "B", "similarity": 0.78, "contains_answer": True},
]

top_result = max(controlled_results, key=lambda item: item["similarity"])
print("가장 높은 점수의 문서:", top_result)
print("실제 정답 포함 여부:", top_result["contains_answer"])
print("→ 유사도 점수는 후보 순위를 정할 뿐, 정답 여부는 본문이나 근거 평가로 확인해야 한다.")


**교정된 접근**: 유사도 점수를 "정답 확률"로 취급해 자동으로 채택하지 않는다. 대신
Top-k로 후보를 좁힌 뒤, 각 후보의 실제 본문을 확인하거나(사람 또는 LLM), Notebook 03-3에서
다룰 "근거 적합성 평가" 단계를 거쳐 정말로 질문에 답할 수 있는 내용인지 검증해야 한다.
8~9번에서 얻은 검색 결과도 순위와 점수만 보지 말고, `text` 필드를 직접 읽어 질문에
대한 답이 맞는지 확인하는 습관이 필요하다.


## 11. 도전 과제

1. `SAMPLE_DOCUMENTS`에 완전히 새로운 주제의 문서를 2~3개 추가하고, 관련 없는
   질문을 던졌을 때 유사도 점수가 어떻게 분포하는지 관찰한다.
2. `search()`에 `section` 필터까지 함께 적용해, `document_id`와 `section`을 동시에
   좁혔을 때 후보 수가 어떻게 줄어드는지 확인한다.
3. Top-k를 1부터 9까지 바꿔가며 같은 질문을 검색하고, 몇 번째 순위부터 유사도
   점수가 급격히 낮아지는지(문서와 무관해지는지) 관찰한다.


## 12. 테스트

**테스트 유형: 외부 API 통합 테스트 — OpenAI Embedding + Chroma**

검색 순위와 metadata filter를 검증한다. 실패하면 코드와 함께 API Key, 네트워크, Embedding 모델과 Chroma Collection 상태를 확인한다.


In [ ]:
assert len(ARTICLES) == 47
assert RETRIEVAL_DOCUMENTS
assert all(doc["document_id"] == LAW_ID for doc in RETRIEVAL_DOCUMENTS)

search_check = search("고영향 인공지능 사업자 의무", k=5, scope="본칙")
assert len(search_check) == 5
assert [result["rank"] for result in search_check] == [1, 2, 3, 4, 5]
assert all(
    search_check[index]["similarity"] >= search_check[index + 1]["similarity"]
    for index in range(len(search_check) - 1)
)
assert all(result["scope"] == "본칙" for result in search_check)

addendum_check = search("시행일", k=len(ADDENDUM_ARTICLES), scope="부칙")
assert len(addendum_check) == len(ADDENDUM_ARTICLES)
assert all(result["scope"] == "부칙" for result in addendum_check)

article34_check = search("위험관리와 이용자 보호", k=10, article="제34조")
assert article34_check
assert all(result["article"] == "제34조" for result in article34_check)

try:
    search("인공지능", k=0)
    raise AssertionError("k=0이 통과했습니다.")
except ValueError:
    pass
print("법률 Embedding Retrieval 테스트 통과")


## 13. 결과 저장


In [ ]:
all_result_rows = []
for label, results in [
    ("baseline", baseline_results),
    ("top1", top1),
    ("top5", top5),
    ("short_query", short_results),
    ("specific_query", specific_results),
    ("legal_term", legal_term_results),
    ("common_term", common_term_results),
    ("without_filter", without_filter),
    ("addendum_only", addendum_only),
]:
    all_result_rows.extend(build_result_dataframe(label, results).to_dict(orient="records"))

retrieval_results_df = pd.DataFrame(all_result_rows)
retrieval_dir = OUTPUT_DIR / "retrieval"
retrieval_dir.mkdir(parents=True, exist_ok=True)
retrieval_csv_path = retrieval_dir / "aitrust_retrieval_results.csv"
retrieval_results_df.to_csv(retrieval_csv_path, index=False, encoding="utf-8-sig")
print("저장 위치:", retrieval_csv_path)

embedding_log = {
    "source_file": SOURCE_FILE,
    "law_name": LAW_NAME,
    "article_count": len(ARTICLES),
    "retrieval_chunk_count": len(RETRIEVAL_DOCUMENTS),
    "vector_store": type(vector_store).__name__,
    "collection": COLLECTION_NAME,
    "baseline_query": BASELINE_QUERY,
    "baseline_top_results": [
        {"rank": r["rank"], "doc_id": r["doc_id"], "article": r["article"], "similarity": round(r["similarity"], 4)}
        for r in baseline_results
    ],
}
saved_path = save_log(
    embedding_log,
    OUTPUT_DIR / "logs" / "aitrust_2_embedding_retrieval_log.json",
)
print("저장 위치:", saved_path)


## 14. 핵심 정리

- Embedding은 텍스트를 벡터로 표현해 의미 기반 검색을 가능하게 한다.
- Vector Store는 문서 Embedding 저장, 질문 Embedding, 유사도 계산, 정렬, Top-k
  선택을 담당하며 애플리케이션은 검색 조건과 결과 활용에 집중한다.
- 유사도 점수는 순위를 매기는 상대적 신호일 뿐, 정답일 확률이 아니다.
- `k`는 Vector Store가 반환할 상위 결과 수를 정하고, Metadata Filter는 유사도 검색
  전에 후보군 자체를 좁힌다.
- 짧은 검색어, 동의어, 동일 키워드의 다른 의미는 모두 검색 결과에 영향을 주며,
  키워드 검색과 Embedding 검색이 서로 다르게 반응한다.
- 검색 결과를 그대로 신뢰하지 않고, 실제 본문을 확인해 정말 질문에 답이 되는지
  검증하는 절차가 필요하다.


## 15. 확인 문제

1. 유사도 점수 0.85가 "정답일 확률 85%"를 의미하지 않는 이유는 무엇인가?
2. 순위 정렬/Top-k와 Metadata Filter는 검색 후보군에 어떻게 다르게 영향을 주는가?
3. 키워드 검색이 "생성형 AI"와 "생성형 인공지능"를 같은 의미로 찾지 못하는 이유는
   무엇인가?
4. 유사도 점수가 높은 검색 결과를 받았을 때, 그것을 바로 정답으로 채택하면 안 되는
   이유는 무엇이며 대신 어떻게 해야 하는가?
